In [13]:
import pandas as pd
import pickle

from sklearn.model_selection import train_test_split

from sklearn.pipeline import Pipeline

from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.linear_model import LogisticRegression

from sklearn.naive_bayes import MultinomialNB

from sklearn.tree import DecisionTreeClassifier

from sklearn.ensemble import RandomForestClassifier

from sklearn.svm import LinearSVC

from sklearn.metrics import accuracy_score

import warnings
warnings.filterwarnings("ignore")

In [14]:
df = pd.read_csv("../data/processed/final_dataset.csv")

df.head()

,clean_text,label
0,target roe v wade oklahoma bill making felony ...,1
1,study woman drive time farther texas law close...,1
2,trump clinton clash dueling dc speech donald t...,1
3,grand jury texas indicts activist behind plann...,1
4,reproductive right hang balance debate moderat...,1


In [15]:
X = df["clean_text"]

y = df["label"]

In [16]:
X_train, X_test, y_train, y_test = train_test_split(

    X,

    y,

    test_size=0.20,

    random_state=42,

    stratify=y

)

In [17]:
models = {

    "Logistic Regression": LogisticRegression(
        max_iter=5000,
        random_state=42
    ),

    "Naive Bayes": MultinomialNB(),

    "Decision Tree": DecisionTreeClassifier(
        max_depth=20,
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        max_depth=25,
        random_state=42,
        n_jobs=-1
    ),

    "SVM": LinearSVC(
        C=1.0,
        max_iter=5000,
        random_state=42
    )

}

In [18]:
trained_models = {}

results = []

In [19]:
for name, classifier in models.items():

    pipeline = Pipeline([

        (

            "tfidf",

            TfidfVectorizer(

                max_features=5000,

                stop_words="english"

            )

        ),

        (

            "classifier",

            classifier

        )

    ])

    pipeline.fit(

        X_train,

        y_train

    )

    prediction = pipeline.predict(

        X_test

    )

    accuracy = accuracy_score(

        y_test,

        prediction

    )

    trained_models[name] = pipeline

    results.append({

        "Model": name,

        "Accuracy": accuracy

    })

In [20]:
results = pd.DataFrame(results)

results.sort_values(

    by="Accuracy",

    ascending=False,

    inplace=True

)

results

,Model,Accuracy
4,SVM,0.946681
0,Logistic Regression,0.928183
3,Random Forest,0.905332
1,Naive Bayes,0.883569
2,Decision Tree,0.845484


In [21]:
import os

os.makedirs("../models", exist_ok=True)

for name, model in trained_models.items():

    filename = (

        name

        .lower()

        .replace(" ","_")

        + ".pkl"

    )

    with open(

        "../models/"+filename,

        "wb"

    ) as file:

        pickle.dump(

            model,

            file

        )

In [22]:
best_model = results.iloc[0]

best_model

Model            SVM
Accuracy    0.946681
Name: 4, dtype: object

In [23]:
results.to_csv(

    "../models/model_results.csv",

    index=False

)

In [24]:
print("="*60)

print("Training Completed")

print("="*60)

print(results)

print("="*60)

print(

    "Best Model:",

    best_model["Model"]

)

Training Completed
                 Model  Accuracy
4                  SVM  0.946681
0  Logistic Regression  0.928183
3        Random Forest  0.905332
1          Naive Bayes  0.883569
2        Decision Tree  0.845484
Best Model: SVM
